# components.page_renderer

> Full session management page assembly — header with title and the session list body.

Cross-page navigation chrome (tab bars, navbars, etc.) is **not** this library's concern — hosts are expected to wrap the rendered page in their own layout if they want site-wide chrome. The session manager page renders its own title + body, nothing more.

In [ ]:
#| default_exp components.page_renderer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, Optional

from fasthtml.common import Div, H1

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import container, max_w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, gap,
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V11 icon-size roles)
from cjm_fasthtml_design_system.icons import icons

from cjm_fasthtml_workflow_session_management.models import SessionManagementUrls
from cjm_fasthtml_workflow_session_management.html_ids import SessionManagerHtmlIds
from cjm_fasthtml_workflow_session_management.components.helpers import DEBUG_SESSION_RENDER

In [ ]:
#| export
def render_page_header(
    title:str="Sessions", # Page title
    icon_name:str="layers", # Lucide icon for the title
) -> Any: # Header element
    """Render the page header with an icon + title."""
    return Div(
        H1(
            lucide_icon(icon_name, size=icons.page_title),
            title,
            cls=combine_classes(
                font_size._3xl, font_weight.bold,
                flex_display, items.center, gap(3),
            ),
        ),
        cls=combine_classes(flex_display, flex_direction.col, m.b(4)),
    )

In [ ]:
h = render_page_header()
assert h.tag == "div"
assert len(h.children) == 1  # just the H1 title
assert h.children[0].tag == "h1"

# Custom title + icon.
h2 = render_page_header(title="Manage", icon_name="database")
assert h2.children[0].children[1] == "Manage"
print("Page header OK")

## Full Page

`render_session_manager_page` takes a late-bound `render_list_fn` so state mutations are reflected at render time (not at closure-capture time).

In [ ]:
#| export
def render_session_manager_page(
    urls:SessionManagementUrls, # URL bundle (unused directly, reserved for future header actions)
    render_list_fn:Callable, # () -> session list component
    title:str="Sessions", # Page title
    icon_name:str="layers", # Lucide icon name for the title
) -> Any: # Complete session manager page component
    """Render the complete session manager page with header and session list."""
    if DEBUG_SESSION_RENDER:
        print(f"[RENDER] session_manager_page")
    return Div(
        render_page_header(title=title, icon_name=icon_name),
        render_list_fn(),
        id=SessionManagerHtmlIds.PAGE_CONTENT,
        cls=combine_classes(
            container, max_w._5xl, m.x.auto,
            flex_display, flex_direction.col,
            p(4), gap(4), h.full,
        ),
    )

In [ ]:
# Late-bound render_list_fn is called at render time, not capture time.
_call_count = [0]
def _list_stub():
    _call_count[0] += 1
    return Div("list content", id="stub-list")

_urls = SessionManagementUrls(
    management_page="/m/sessions/", list_sessions="/m/sessions/list",
    session_detail="/m/sessions/detail", create_session="/m/sessions/create",
    delete_session="/m/sessions/delete", rename_session="/m/sessions/rename",
    resume_session="/m/sessions/resume",
)
page = render_session_manager_page(_urls, _list_stub)
assert _call_count[0] == 1, "render_list_fn should be called exactly once per render"
assert page.tag == "div"
assert page.attrs["id"] == SessionManagerHtmlIds.PAGE_CONTENT
assert page.children[1].attrs["id"] == "stub-list"
print(f"Page OK: {len(page.children)} children")

Page OK: 2 children


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()